# Preconditioning interface for stationary control problems

So far, we have employed the in-built preconditioners for solving the optimal control problem at hand. In this notebook, we describe how to apply a different preconditioner. We begin considering the stationary case, giving as an example a linear Poisson control problem, then describe how to apply the preconditioner to the case of instationary control problems.

## Preconditioning Poisson control problems

We consider the following Poisson control problem:

$$
\min_{v,u} \frac{1}{2} \| v - v_d \| ^2 + \frac{1}{\beta} \| u \| ^2$$

subject to:

$$
    -\nabla^2 v = f + u \qquad \mathrm{in} \; \Omega := (-1, 1)^2, 
$$
$$
    v = 1  \qquad \mathrm{on} \; \partial \Omega,
$$

with $\beta = 10^{-4}$. We set $f = \frac{\pi^2}{2} \cos(\frac{\pi x_1}{2}) \cos(\frac{\pi x_2}{2})$, and seek the desired state $v_d = \cos(\frac{\pi x_1}{2}) \cos(\frac{\pi x_2}{2}) + 1$. For this setting, the analytical state solution is given by $v = v_d$, while the adjoint variable is $\zeta = 0$.

We begin by defining the problem with the control class.

In [ ]:
from firedrake import *
from control.control import *

mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)
space_0 = FunctionSpace(mesh, "Lagrange", 1)
beta = 1.0e-4


# the desired state
def desired_state(test):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)
    x_1 = X[0]
    x_2 = X[1]

    v_d = Function(space, name="v_d")
    v_d.interpolate(cos(0.5 * pi * x_1) * cos(0.5 * pi * x_2) + 1.0)

    return inner(v_d, test) * dx, v_d


# the force function
def force_f(test):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)
    x_1 = X[0]
    x_2 = X[1]

    f = Function(space, name="f")
    f.interpolate(0.5 * pi * pi * cos(0.5 * pi * x_1) * cos(0.5 * pi * x_2))

    return inner(f, test) * dx


# the boundary conditions
bcs_v = DirichletBC(space_0, 1.0, "on_boundary")


# the forward form
def forw_diff_operator(trial, test, v):
    return inner(grad(trial), grad(test)) * dx


Poisson_control = Control.Stationary(
    space_0, forw_diff_operator, desired_state=desired_state,
    force_function=force_f, beta=beta, bcs_v=bcs_v)

We now have to define the preconditioner we would like to apply. We will employ Zulehner's preconditioner for the Poisson control problem, introduced in [1].

The preconditioner is constructed by means of a callable that takes as input the control object, the adjoint form D_zeta, the forward form D_v, the (homogenized) boundary conditions on the state variable bcs_v, and the boundary conditions on the adjoint variable bcs_zeta. The $(1,1)$-block of the system to be solved is stored in the control's module _M_v.

In [ ]:
# convergence test for the inner linear solvers
def converged(ksp, it, rnorm):
    return it >= ksp.max_it


# Zulehner's preconditioner for the Poisson control problem
def P(self, D_zeta, D_v, bcs_v, bcs_zeta):
    sp = {"ksp_type": "preonly",
          "pc_type": "hypre",
          "pc_hypre_type": "boomeramg",
          "ksp_max_it": 1,
          "pc_hypre_boomeramg_max_iter": 2,
          "ksp_atol": 0.0,
          "ksp_rtol": 0.0}

    solver_0 = LinearSolver(
        assemble(self._M_v + (beta**0.5) * D_zeta, bcs=bcs_v),
        solver_parameters=sp)

    solver_1 = LinearSolver(
        assemble(self._M_v + (beta**0.5) * D_v, bcs=bcs_zeta),
        solver_parameters=sp)

    solver_0.ksp.addConvergenceTest(converged, prepend=True)
    solver_1.ksp.addConvergenceTest(converged, prepend=True)

    # definition of preconditioner
    def pc_linear(u_0, u_1, b_0, b_1):
        # applying Zulehner's preconditioner

        # solving for the (1,1)-block
        u_0.zero()
        solver_0.solve(u_0, b_0.copy(deepcopy=True))

        # solving for the (2,2)-block
        u_1.zero()
        solver_1.solve(u_1, b_1.copy(deepcopy=True))
        with u_1.dat.vec as u:
            u.scale(beta)

    return pc_linear

Since the preconditioner is symmetric positive definite, we can employ MINRES as linear solver.

In [ ]:
solver_parameters = {"linear_solver": "minres",
                     "maximum_iterations": 500,
                     "relative_tolerance": 1.0e-8,
                     "absolute_tolerance": 1.0e-8,
                     "monitor_convergence": True}

We can now call the linear_solve() module, passing the function that construct the preconditioner to the kwarg P, and then check the numerical error, as follows.

In [ ]:
Poisson_control.linear_solve(
    P=P, solver_parameters=solver_parameters,
    create_output=False, plots=False)

v = Poisson_control._v
zeta = Poisson_control._zeta

true_v = Function(space_0)
true_zeta = Function(space_0)

# evaluating the true solution
X = SpatialCoordinate(mesh)
x_1 = X[0]
x_2 = X[1]

true_v.interpolate(cos(0.5 * pi * x_1) * cos(0.5 * pi * x_2) + 1.0)
true_zeta.zero()

# evaluating the norm of the error
v_error = np.sqrt(abs(assemble(
    inner(v - true_v, v - true_v) * dx)))
zeta_error = np.sqrt(abs(assemble(
    inner(zeta - true_zeta, zeta - true_zeta) * dx)))

print(f"{v_error=}")
print(f"{zeta_error=}")

In case of non-linear problems, if one wishes to employ a user-built preconditioner in the linear solver, one has to pass the callable that builds the preconditioner to the kwarg P in the non_linear_solve() module.


## Preconditioning stationary incompressible Stokes control problems

We now consider the control of the incompressible Stokes equations. The preconditioning infrastructure works roughly as the case of Poisson control problems, with the user that has to provide a routine for the application of the preconditioner. The routine has to take as input the same ones for the previous problem, with the addition of the nullspaces for the state and velocity variables.

To give an example, we consider the following Stokes control problem:

$$
\min_{\vec{v},\vec{u}} \frac{1}{2} \| \vec{v} - \vec{v}_d \| ^2 + \frac{1}{\beta} \| \vec{u} \| ^2
$$

subject to:

$$
    - \nabla^2 \; \vec{v} + \nabla p = \vec{f} + \vec{u} \qquad \mathrm{in} \; \Omega := (-1, 1)^2, 
$$
$$
    - \nabla \cdot \vec{v} = 0 \qquad \mathrm{in} \; \Omega,
$$
$$
    \vec{v} = \vec{g} \qquad \mathrm{on} \; \partial \Omega.
$$

We consider the lid-driven cavity problem in $\Omega = (-1, 1)^2$, with force function $\vec{f}=[0, 0]^\top$ and boundary conditions given by

$$
    \vec{g} = [0, 1]^\top \quad \mathrm{on} \; \partial \Omega_1:=(-1,1) \times \{ 1\},
$$
$$
    \vec{g} = [0, 0]^\top \quad \mathrm{on} \; \partial \Omega \setminus \partial \Omega_1.
$$

We seek $\vec{v}_d = [0, 0]^\top$ as desired state, and set $\beta = 10^{-4}$. We first construct the problem.

In [ ]:
from firedrake import *
from control.preconditioner import ConstantNullspace
from control.control import *

mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)

beta = 1.0e-4

space_v = VectorFunctionSpace(mesh, "Lagrange", 2)
space_p = FunctionSpace(mesh, "Lagrange", 1)

bcs_v = [DirichletBC(space_v, Constant((1., 0.)), 4),
         DirichletBC(space_v, Constant((0., 0.)), (1, 2, 3))]


def forw_diff_operator(trial, test, v):
    # spatial differential for the forward problem
    return inner(grad(trial), grad(test)) * dx


def desired_state(test):
    space = test.function_space()

    # desired state
    v_d = Function(space)
    v_d.zero()

    return inner(v_d, test) * dx, v_d


def force_f(test):
    space = test.function_space()

    # force function
    f = Function(space)
    f.zero()

    return inner(f, test) * dx


stationary_Stokes_control = Control.Stationary(
    space_v, forw_diff_operator, desired_state=desired_state,
    force_function=force_f, beta=beta, space_p=space_p, bcs_v=bcs_v)

We now build the spd preconditioner described in [1].

In [ ]:
def converged(ksp, it, rnorm):  # noqa: F811
    return it >= ksp.max_it


def P(self, D_zeta, D_v, B, nullspace_v, nullspace_zeta, bcs_v, bcs_zeta):
    space_v = self._space_v
    space_p = self._space_p
    p_test, p_trial = TestFunction(space_p), TrialFunction(space_p)

    M_p = inner(p_trial, p_test) * dx
    K_p = inner(grad(p_trial), grad(p_test)) * dx

    # multigrid for the (1,1)-block
    sp_11block = {"ksp_type": "preonly",
                  "pc_type": "hypre",
                  "pc_hypre_type": "boomeramg",
                  "ksp_max_it": 1,
                  "pc_hypre_boomeramg_max_iter": 2,
                  "ksp_atol": 0.0,
                  "ksp_rtol": 0.0}

    # employing Chebyshev for the pressure-mass matrix
    e_min_p = 0.5
    e_max_p = 2.0
    sp_M_p = {
        "ksp_type": "chebyshev",
        "pc_type": "jacobi",
        "ksp_chebyshev_eigenvalues": f"{e_min_p:.16e}, {e_max_p:.16e}",
        "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
        "ksp_chebyshev_esteig_steps": 0,
        "ksp_chebyshev_esteig_noisy": False,
        "ksp_max_it": 20,
        "ksp_atol": 0.0,
        "ksp_rtol": 0.0}

    # employing multigrid for the pressure-stiffness matrix
    sp_K_p = {"ksp_type": "preonly",
              "pc_type": "hypre",
              "pc_hypre_type": "boomeramg",
              "ksp_max_it": 1,
              "pc_hypre_boomeramg_max_iter": 1,
              "ksp_atol": 0.0,
              "ksp_rtol": 0.0}

    solver_0 = LinearSolver(
        assemble(self._M_v + (beta**0.5) * D_zeta, bcs=bcs_v),
        solver_parameters=sp_11block)
    solver_0.ksp.addConvergenceTest(converged, prepend=True)

    solver_1 = LinearSolver(
        assemble(self._M_v + (beta**0.5) * D_v, bcs=bcs_zeta),
        solver_parameters=sp_11block)
    solver_1.ksp.addConvergenceTest(converged, prepend=True)

    solver_M_p = LinearSolver(
        assemble(M_p), solver_parameters=sp_M_p)
    solver_M_p.ksp.addConvergenceTest(converged, prepend=True)

    solver_K_p = LinearSolver(
        assemble(K_p), solver_parameters=sp_K_p)
    solver_K_p.ksp.addConvergenceTest(converged, prepend=True)

    # definition of preconditioner
    def pc_linear(u_0, u_1, b_0, b_1):
        # applying Zulehner's preconditioner

        b_00 = Cofunction(space_v.dual())
        b_01 = Cofunction(space_v.dual())

        b_00.assign(b_0.sub(0))
        b_01.assign(b_0.sub(1))

        # solving for the (1,1)-block
        u_0.sub(0).zero()
        solver_0.solve(u_0.sub(0), b_00.copy(deepcopy=True))

        # solving for the (2,2)-block
        u_0.sub(1).zero()
        solver_1.solve(u_0.sub(1), b_01.copy(deepcopy=True))
        with u_0.sub(1).dat.vec as u:
            u.scale(beta)

        del b_00
        del b_01

        b_10 = Cofunction(space_p.dual())
        b_11 = Cofunction(space_p.dual())

        # solving for the (3,3)-block
        b_10.assign(b_1.sub(0))
        u_1.sub(0).zero()
        solver_K_p.solve(u_1.sub(0), b_10.copy(deepcopy=True))

        b = Function(space_p)
        b.zero()
        solver_M_p.solve(b, b_10.copy(deepcopy=True))
        with b.dat.vec as b_v:
            b_v.scale(beta**0.5)

        with u_1.sub(0).dat.vec as b_v, \
                b.dat.vec_ro as b_1_v:
            b_v.axpy(1.0, b_1_v)
        del b

        # solving for the (4,4)-block
        b_11.assign(b_1.sub(1))
        u_1.sub(1).zero()
        solver_K_p.solve(u_1.sub(1), b_11.copy(deepcopy=True))

        b = Function(space_p)
        b.zero()
        solver_M_p.solve(b, b_11.copy(deepcopy=True))
        with b.dat.vec as b_v:
            b_v.scale(beta**0.5)

        with u_1.sub(1).dat.vec as b_v, \
                b.dat.vec_ro as b_1_v:
            b_v.axpy(1.0, b_1_v)
        del b

        with u_1.sub(1).dat.vec as b_v:
            b_v.scale(1.0 / beta)

        del b_10
        del b_11

    return pc_linear

Since the preconditioner is symmetric positive definite, we run MINRES as linear solver up to a reduction of $10^{-8}$ on either the absolute or the relative residual. Then, we call the module instationary_linear_solve(), passing the preconditioner as kwarg to the call.

In [ ]:
solver_parameters = {
    "linear_solver": "minres",
    "maximum_iterations": 200,
    "relative_tolerance": 1.0e-08,
    "absolute_tolerance": 1.0e-08,
    "monitor_convergence": True}

stationary_Stokes_control.incompressible_linear_solve(
    ConstantNullspace(), P=P, solver_parameters=solver_parameters,
    create_output=False, plots=False)

As above, in case of non-linear problems, if one wishes to employ a user-built preconditioner in the linear solver, one has to pass the callable that builds the preconditioner to the kwarg P in the non_linear_solve() module.

[1] Zulehner W.: Nonstandard Norms and Robust Estimates for Saddle Point Problems, SIAM J. Matrix Anal. Appl. 32, 536–560, 2011.